# Day3_01E_Multi_Agent_Handoffs

## OpenAI Agents SDK - Teaching Notebook

**Duration:** 20 minutes

### Learning Objectives

- Understand why one Agent is not enough for complex problems.
- Learn the concept of specialist Agents.
- Build a simple multi-agent workflow.
- Understand Agent handoffs.
- Relate the concept to enterprise and university scenarios.


## 1. Why One Agent is Not Enough?

Imagine a student asks:

> "Create a lesson plan on Agentic AI, prepare a quiz and recommend references."

Should one Agent do everything?

It could.

But specialists generally perform better.

Just like in a university:

- Faculty teaches.
- Librarian manages books.
- Exam Cell prepares assessments.
- HOD approves curriculum.

This is exactly how multi-agent systems work.


In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import os

from agents import Agent, Runner

current = Path.cwd().resolve()

for folder in [current] + list(current.parents):
    env = folder / ".env"
    if env.exists():
        load_dotenv(env)
        break

print("API Key Loaded:", os.getenv("OPENAI_API_KEY") is not None)

## 2. Create Specialist Agents

In [ ]:
lesson_agent = Agent(
    name="Lesson Planner",
    instructions="""
    Create simple classroom-friendly lesson plans for engineering faculty.
    """
)

quiz_agent = Agent(
    name="Quiz Creator",
    instructions="""
    Create short quizzes with answers for engineering students.
    """
)

reference_agent = Agent(
    name="Reference Expert",
    instructions="""
    Recommend books, papers and online resources for learning.
    """
)

### Pause & Reflect

Each Agent has a single responsibility.

This makes prompts simpler, easier to maintain and often improves response quality.


## 3. Run Each Agent Individually

In [ ]:
lesson = await Runner.run(
    lesson_agent,
    "Create a 30-minute lesson plan on Agentic AI."
)

print(lesson.final_output)

In [ ]:
quiz = await Runner.run(
    quiz_agent,
    "Create five MCQs on Agentic AI."
)

print(quiz.final_output)

In [ ]:
references = await Runner.run(
    reference_agent,
    "Suggest beginner resources on Agentic AI."
)

print(references.final_output)

## 4. Simulating a Handoff

A coordinator Agent (or application) decides which specialist should perform the task.

For teaching purposes, we'll simulate this routing with Python.


In [ ]:
async def coordinator(user_request: str):

    request = user_request.lower()

    if "lesson" in request:
        result = await Runner.run(lesson_agent, user_request)

    elif "quiz" in request:
        result = await Runner.run(quiz_agent, user_request)

    elif "reference" in request or "book" in request:
        result = await Runner.run(reference_agent, user_request)

    else:
        return "No suitable specialist found."

    return result.final_output

## 5. Test the Coordinator

In [ ]:
print(await coordinator(
    "Create a lesson plan on RAG."
))

In [ ]:
print(await coordinator(
    "Prepare a quiz on Function Tools."
))

In [ ]:
print(await coordinator(
    "Recommend books on Agentic AI."
))

### What Did We Just Learn?

The coordinator doesn't solve the problem itself.

Its responsibility is to route the request to the best specialist.


## 6. Enterprise Example

A banking assistant may contain:

- Customer Support Agent
- Loan Eligibility Agent
- Fraud Detection Agent
- Compliance Agent
- Payment Agent

The customer sees one assistant, but multiple specialist Agents collaborate behind the scenes.


## 7. DSU Example

Faculty asks:

> "Prepare tomorrow's AI class."

Possible flow:

Faculty Request

↓

Lesson Planning Agent

↓

Quiz Agent

↓

Reference Agent

↓

Combined Response


In [ ]:
async def prepare_class(topic):

    lesson = await Runner.run(
        lesson_agent,
        f"Create a lesson plan on {topic}"
    )

    quiz = await Runner.run(
        quiz_agent,
        f"Create five MCQs on {topic}"
    )

    refs = await Runner.run(
        reference_agent,
        f"Suggest references for {topic}"
    )

    return f'''
===== LESSON PLAN =====

{lesson.final_output}

===== QUIZ =====

{quiz.final_output}

===== REFERENCES =====

{refs.final_output}
'''

## 8. Complete Demo

In [ ]:
output = await prepare_class("Agentic AI")

print(output)

## Hands-on Exercise

Create a fourth specialist Agent called **Assignment Creator**.

Then update the coordinator so it can also route assignment requests.


## Challenge Exercise

Modify `prepare_class()` so that all specialist outputs are combined into a formatted Markdown report suitable for sharing with students.


# Key Takeaways

- Complex tasks are easier when split among specialist Agents.
- Handoffs improve modularity and maintainability.
- A coordinator routes work to the most suitable Agent.
- Multi-Agent systems mirror how teams collaborate in universities and enterprises.
- This pattern is the foundation for frameworks like CrewAI and LangGraph.
